In [11]:
# Standard library
import json
import warnings
from collections import Counter
from pathlib import Path

# Arrays, tables and the association statistics of the relevance table
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, pointbiserialr

# The imputers of the recipe, the model, and the selectors compared in this notebook
from sklearn.feature_selection import RFE, SelectFromModel, SelectKBest, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# The group's own functions (preprocessing.py, beside this notebook)
from preprocessing import (RANDOM_STATE, columns_type, fill_missing, fold_matrices, held_out_ap,
                           load_data, paired_delta, target, temporal_splits)

In [12]:
train_clientes = load_data()
numeric, categorical = columns_type(train_clientes)
frame = train_clientes[numeric + categorical + [target]]
time_splits = temporal_splits(frame)

# The pre-processing team's decisions (1.7.2 and 1.8), applied as rules inside each split
labels_as_numbers = ["is_first_purchase", "is_weekend", "since_2015_or_earlier", "purchase_year",
                     "purchase_quarter", "purchase_month", "purchase_day", "purchase_dayofweek",
                     "customer_since_year"]
measures = [c for c in numeric if c not in labels_as_numbers]
# log1p only where no value is negative, as in 1.7.2
log1p_columns = [c for c in measures if not (train_clientes[c] < 0).any()]


def make_recipe(numeric, categorical):
    """The final recipe for a given set of columns."""
    return {"numeric": numeric, "categorical": categorical,
            "fill": [(numeric, SimpleImputer(strategy="median")),
                     (categorical, SimpleImputer(strategy="most_frequent"))],
            "log1p": [c for c in log1p_columns if c in numeric],
            "rare": {"transport_method": 50, "payment_type": 20,
                     "city": 30, "municipality": 30, "postal_area": 20}}


recipe = make_recipe(numeric, categorical)
warnings.filterwarnings("ignore", category=UserWarning)  # unknown levels in validation, as the professor does

Before moving on to the feature selection techniques, we need to set a **reference** for the results given by our metric (**AP** -> average precision), so that we can then compare them with the results obtained when the techniques are applied.

The function `held_out_ap` receives four parameters:

1. `frame` (the data)
2. `time_splits` (the 7 temporal splits: each one trains on the past and validates on the period right after it)
3. `recipe` (The pre-processing methods defined in the previus notebooks)
4. `select` (receives the feature selection technique to apply inside each split)

> **Note:** the reference level leaves it empty, meaning no feature selection technique was used.

After getting these, for each split it will prepare the training and validation rows with the recipe, finally trains a logistic regression on the training rows and scores it on the validation rows.

In [13]:
reference_all = held_out_ap(frame, time_splits, recipe)
print(reference_all[["n", "AP", "F1"]])
print(f"mean: AP {reference_all['AP'].mean():.4f} | F1 {reference_all['F1'].mean():.4f}")

     n        AP        F1
0  144  0.827941  0.702341
1  159  0.837289  0.696203
2  181  0.859126  0.738137
3  193  0.831941  0.725490
4  209  0.806782  0.701068
5  223  0.871501  0.749621
6  235  0.858898  0.770328
mean: AP 0.8419 | F1 0.7262


---

## 1. Id-like variables

In [14]:
# How many customers stand behind each level, before and after the rare-level grouping
identifier_like = ["city", "postal_area", "municipality", "transport_method"]
for column in identifier_like:
    per_level = train_clientes.dropna(subset=[column]).groupby(column)["customer_id"].nunique()
    counts = train_clientes[column].value_counts()
    grouped = per_level[counts[counts >= recipe["rare"][column]].index]
    print(f"{column:17s} {per_level.size:3d} levels, median {per_level.median():.0f} customers, {int((per_level == 1).sum()):3d} with one"
          f" | after grouping: {grouped.size:2d} levels, median {grouped.median():.0f}, {int((grouped == 1).sum()):2d} with one")

# The model without them
without_ids = [c for c in categorical if c not in identifier_like]
run = held_out_ap(frame, time_splits, make_recipe(numeric, without_ids))
print(f"without identifier-like: {run['n'].mean():.0f} columns | AP {run['AP'].mean():.4f} | F1 {run['F1'].mean():.4f}")
for metric in ("AP", "F1"):
    r = paired_delta(run[metric], reference_all[metric])
    print(f"   d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")


# random_noise: its relevance on split 0, and the model with it
noisy = train_clientes[numeric + ["random_noise"] + categorical + [target]]
train_rows, _ = time_splits[0]
r = pointbiserialr(noisy.iloc[train_rows][target], noisy.iloc[train_rows]["random_noise"]).statistic
print(f"random_noise: |point-biserial r| on split 0 = {abs(r):.4f}")
run = held_out_ap(noisy, time_splits, make_recipe(numeric + ["random_noise"], categorical))
print(f"with random_noise: {run['n'].mean():.0f} columns | AP {run['AP'].mean():.4f} | F1 {run['F1'].mean():.4f}")
for metric in ("AP", "F1"):
    r = paired_delta(run[metric], reference_all[metric])
    print(f"   d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")

city              259 levels, median 1 customers, 220 with one | after grouping: 38 levels, median 2, 17 with one
postal_area       189 levels, median 1 customers, 109 with one | after grouping: 48 levels, median 4,  5 with one
municipality      151 levels, median 2 customers,  67 with one | after grouping: 37 levels, median 6,  0 with one
transport_method   80 levels, median 2 customers,  40 with one | after grouping: 23 levels, median 25,  0 with one
without identifier-like: 93 columns | AP 0.8456 | F1 0.7419
   dAP +0.0036 +/- 0.0032, better on 5/7 -> inside the noise
   dF1 +0.0157 +/- 0.0037, better on 6/7 -> better
random_noise: |point-biserial r| on split 0 = 0.0054
with random_noise: 193 columns | AP 0.8417 | F1 0.7286
   dAP -0.0003 +/- 0.0002, better on 2/7 -> inside the noise
   dF1 +0.0025 +/- 0.0016, better on 5/7 -> inside the noise


In [15]:
# From here on, every technique starts from the candidate columns
categorical = without_ids
frame = train_clientes[numeric + categorical + [target]]
recipe = make_recipe(numeric, categorical)
reference_candidates = held_out_ap(frame, time_splits, recipe)
print(f"reference_candidates: {reference_candidates['n'].mean():.0f} columns | AP {reference_candidates['AP'].mean():.4f}"
      f" | F1 {reference_candidates['F1'].mean():.4f}")

reference_candidates: 93 columns | AP 0.8456 | F1 0.7419


---

## 2. Feature Selection

This section compares feature selection techniques from the three families seen in class (filter, wrapper and embedded) on the 93 candidate columns. Every technique is refitted inside each training split and compared with `reference_candidates` on the same 7 splits.

---

### 2.1 Filter Methods

The filters below score the columns without fitting a model: first their variance, then their association with the target.

---

#### 2.1.1 Variance Thresholding

Variance Thresholding removes features whose variance is below a threshold. With a threshold of 0 it only removes constant features, which carry no information. Variance measures spread, not relevance, so a low-variance feature is not necessarily useless.

In [16]:
# Split 0's training matrix, prepared by the recipe
train_rows, validation_rows = time_splits[0]
A_train, _, names = fold_matrices(frame.iloc[train_rows], frame.iloc[validation_rows], recipe)

variances = pd.Series(A_train.var(axis=0), index=names)
measured = variances.index.isin(numeric)
print(f"{len(variances)} columns, {int((variances == 0).sum())} constant")
print(f"dummies : variance {variances[~measured].min():.5f} to {variances[~measured].max():.4f}")
print(f"measured: variance {variances[measured].min():.4f} to {variances[measured].max():.4f}")

81 columns, 0 constant
dummies : variance 0.00045 to 0.2202
measured: variance 1.0000 to 1.0000


In [17]:
variance = held_out_ap(frame, time_splits, recipe, select=lambda: VarianceThreshold(0.0))
print(variance[["n", "AP", "F1"]])
for metric in ("AP", "F1"):
    r = paired_delta(variance[metric], reference_candidates[metric])
    print(f"   d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")

     n        AP        F1
0   81  0.840286  0.722222
1   82  0.841042  0.717629
2   88  0.853993  0.762393
3   91  0.826788  0.723473
4  101  0.823980  0.723183
5  104  0.873739  0.767857
6  104  0.859155  0.776671
   dAP +0.0000 +/- 0.0000, better on 0/7 -> identical on every split
   dF1 +0.0000 +/- 0.0000, better on 0/7 -> identical on every split


**Conclusions:** No column is constant (0 of 81 in split 0), so `VarianceThreshold(0)` removed nothing and its results are identical to `reference_candidates` on every split. Variance does not rank the columns either: after scaling, every numeric column has a variance of exactly 1, and a dummy's variance only reflects how common its category is (0.0004 to 0.22).

#### 2.1.2 Select k-Best

First, each source column is scored against the target on split 0's training rows; then `SelectKBest` keeps the k best encoded columns inside every split.

In [18]:
def cramers_v(column, outcome):
    """Chi-square on a contingency table, rescaled so columns compare.

    Returns the statistic, its p-value, Cramer's V, and the smallest expected
    count. The p-value is an approximation that needs every expected count above
    about 5, so a level almost nobody occupies breaks it. V is still readable as
    an effect size.
    """
    table = pd.crosstab(column, outcome)
    if min(table.shape) < 2:
        return 0.0, float("nan"), 0.0, float(table.to_numpy().min())
    statistic, p, _, expected = chi2_contingency(table, correction=False)
    v = np.sqrt(statistic / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return statistic, p, v, expected.min()


def typed_relevance(rows, outcome):
    """One comparable number per SOURCE column, each by the index its type allows.

    |point-biserial r| for a measurement, Cramer's V for a categorical. Both are
    on [0, 1], so a categorical with many levels and a numeric column can be
    ranked together. A measurement is read as the recipe leaves it, because that
    is the column the model will see.
    """
    scores = {}
    for column in numeric:
        values = rows[column].to_numpy(dtype=float)
        if column in recipe["log1p"]:
            values = np.log1p(values)
        scores[column] = abs(
            pointbiserialr(outcome.to_numpy(dtype=int), values).statistic)
    for column in categorical:
        scores[column] = cramers_v(rows[column].astype(str), outcome)[2]
    return pd.Series(scores).sort_values(ascending=False)

In [19]:
# Split 0's training rows, filled by the recipe
train_rows, validation_rows = time_splits[0]
first_rows, _ = fill_missing(frame.iloc[train_rows], frame.iloc[validation_rows], recipe["fill"])
first_y = first_rows[target]

source_scores = typed_relevance(first_rows, first_y)
table = []
for column in source_scores.index:
    if column in numeric:
        table.append({"column": column, "index": "|point-biserial r|",
                      "value": source_scores[column]})
    else:
        statistic, p, v, expected = cramers_v(first_rows[column].astype(str), first_y)
        table.append({"column": column, "index": "Cramer's V", "value": v,
                      "levels": first_rows[column].nunique(), "chi2": statistic,
                      "p": p, "min expected": expected})
print(pd.DataFrame(table).round(4).to_string(index=False, na_rep=""))

                        column              index  value  levels    chi2      p  min expected
            purchase_count_90d |point-biserial r| 0.3804                                     
           purchase_count_365d |point-biserial r| 0.3708                                     
           purchase_count_180d |point-biserial r| 0.3696                                     
                     spend_90d |point-biserial r| 0.3539                                     
          prior_purchase_count |point-biserial r| 0.3167                                     
                    spend_180d |point-biserial r| 0.3061                                     
                    spend_365d |point-biserial r| 0.2809                                     
             prior_total_spend |point-biserial r| 0.2622                                     
            purchase_count_30d |point-biserial r| 0.2596                                     
      prior_std_purchase_value |point-biserial r| 0.2444    

**Conclusions:**
- Without the identifier-like columns, the customer's purchase volume dominates the ranking (`purchase_count_90d`, `purchase_count_365d`, `purchase_count_180d`, `spend_90d`, `prior_purchase_count`, |r| 0.32–0.38): one fact recorded in several windows.
- The strongest categorical, `commercial_region` (V 0.20), only ranks 13th. Its smallest expected count is 0.44, as for `payment_type`, and 1.31 for `salesperson`, so their p-values cannot be read; only `warehouse` (V 0.05, p 0.02) and `document_series` (V 0.03, p 0.21) have expected counts high enough.
- The categoricals are scored before the recipe groups their rare levels, so these values describe the raw columns rather than exactly what the model sees.

In [20]:
kbest_runs = {}
for k in (10, 20, 50):
    kbest_runs[k] = held_out_ap(frame, time_splits, recipe, select=lambda: SelectKBest(f_classif, k=k))
    print(f"SelectKBest(k={k}): AP {kbest_runs[k]['AP'].mean():.4f} | F1 {kbest_runs[k]['F1'].mean():.4f}")
    for metric in ("AP", "F1"):
        r = paired_delta(kbest_runs[k][metric], reference_candidates[metric])
        print(f"   d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")

# Columns k=10 kept in every split
kept = Counter(c for columns in kbest_runs[10]["kept"] for c in columns)
print([str(c) for c, n in kept.items() if n == len(time_splits)])

SelectKBest(k=10): AP 0.8317 | F1 0.7268
   dAP -0.0139 +/- 0.0036, better on 0/7 -> worse
   dF1 -0.0152 +/- 0.0061, better on 1/7 -> worse
SelectKBest(k=20): AP 0.8464 | F1 0.7473
   dAP +0.0008 +/- 0.0033, better on 3/7 -> inside the noise
   dF1 +0.0054 +/- 0.0092, better on 6/7 -> inside the noise
SelectKBest(k=50): AP 0.8468 | F1 0.7433
   dAP +0.0012 +/- 0.0012, better on 4/7 -> inside the noise
   dF1 +0.0014 +/- 0.0025, better on 4/7 -> inside the noise
['prior_purchase_count', 'prior_total_spend', 'purchase_count_30d', 'purchase_count_90d', 'spend_90d', 'purchase_count_180d', 'spend_180d', 'purchase_count_365d', 'spend_365d']


**Conclusions:**
- With 10 columns the filter is clearly worse than the 93 candidate columns (AP −0.0139, worse on all 7 splits; F1 −0.0152): the columns it keeps in every split are nine purchase-volume columns, one fact recorded many times, because it scores one column at a time.
- With 20 or 50 columns it matches the reference (every difference inside the noise), so most of the 93 columns can be dropped at no measurable cost. Three values of k were tried on the same splits, so the best of them is optimistic.

---

### 2.2 Wrapper Methods

`RFE` fits a logistic regression, removes the column with the smallest coefficient, and repeats until k columns are left. `RFECV` and `SequentialFeatureSelector` are not used: both run their own internal cross-validation, which here would be random rather than temporal, and the preparation would have to be refitted inside every internal fold. With `RFE`, the number of columns k is chosen outside the selector, as in the course practical.

In [21]:
rfe = {}
for k in (10, 20, 50):
    # step=1 removes one column per round, as in the course practical
    rfe[k] = held_out_ap(frame, time_splits, recipe, select=lambda: RFE(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        n_features_to_select=k, step=1))
    print(f"RFE(k={k}): AP {rfe[k]['AP'].mean():.4f} | F1 {rfe[k]['F1'].mean():.4f}")
    for metric in ("AP", "F1"):
        r = paired_delta(rfe[k][metric], reference_candidates[metric])
        print(f"   d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")

# Columns k=20 kept in every split
kept = Counter(c for columns in rfe[20]["kept"] for c in columns)
print([str(c) for c, n in kept.items() if n == len(time_splits)])

RFE(k=10): AP 0.8349 | F1 0.7350
   dAP -0.0106 +/- 0.0055, better on 1/7 -> inside the noise
   dF1 -0.0070 +/- 0.0059, better on 2/7 -> inside the noise
RFE(k=20): AP 0.8397 | F1 0.7327
   dAP -0.0059 +/- 0.0027, better on 2/7 -> worse
   dF1 -0.0092 +/- 0.0064, better on 1/7 -> inside the noise
RFE(k=50): AP 0.8444 | F1 0.7397
   dAP -0.0012 +/- 0.0016, better on 4/7 -> inside the noise
   dF1 -0.0022 +/- 0.0029, better on 2/7 -> inside the noise
['payment_type_Maybe Tomorrow Payment 9', 'salesperson_VLOOKUP Master 8', 'warehouse_Final Excel Warehouse 3', 'commercial_region_Region RG36: We Will See Later', 'customer_tenure_days', 'purchase_count_365d']


**Conclusions:**
- RFE never improves on the 93 candidate columns: with 50 columns it matches the reference (AP −0.0012, F1 −0.0022, inside the noise), with 20 it is worse in AP (−0.0059, better on only 2 of the 7 splits), and with 10 it loses 0.0106 AP, better on only 1 split.
- Its choice is unstable: with k=20, only 6 columns are kept in all 7 splits, against 17 for `SelectKBest` and for L1, and four of them are single dummies (a payment type, a salesperson, a warehouse and a region).
- A likely reason is that RFE ranks the columns by the size of their coefficient, and the coefficients of the 0/1 dummies are not on the scale of the standardized numeric columns, so a rare category can look more important than it is.
- The three values of k were tried on the same splits, so these results are model-selection validation and slightly optimistic, as in the course practical.

### 2.3 Embedded Methods

A logistic regression with an L1 penalty sets the coefficients of weak columns to exactly zero. `SelectFromModel` keeps the columns whose coefficient is not zero, and a plain logistic regression is then trained on them, as for the other techniques. The penalty strength is fixed at C=0.1, the value used in the course practical, rather than tuned.

In [22]:
l1 = held_out_ap(frame, time_splits, recipe, select=lambda: SelectFromModel(
    LogisticRegression(l1_ratio=1, solver="liblinear", C=0.1, max_iter=1000,
                       random_state=RANDOM_STATE),
    threshold=1e-8))
print(f"L1 (C=0.1): {l1['n'].mean():.0f} columns | AP {l1['AP'].mean():.4f} | F1 {l1['F1'].mean():.4f}")
for metric in ("AP", "F1"):
    r = paired_delta(l1[metric], reference_candidates[metric])
    print(f"   d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")

# Columns L1 kept in every split
kept = Counter(c for columns in l1["kept"] for c in columns)
print([str(c) for c, n in kept.items() if n == len(time_splits)])

L1 (C=0.1): 29 columns | AP 0.8459 | F1 0.7402
   dAP +0.0003 +/- 0.0011, better on 3/7 -> inside the noise
   dF1 -0.0017 +/- 0.0030, better on 2/7 -> inside the noise
['island_Island IS1: The Pivot Archipelago', 'Total Imp_', 'Total Desc_ Linha', 'N_ Detalhes', 'days_since_previous_purchase', 'customer_tenure_days', 'prior_purchase_count', 'prior_mean_purchase_value', 'prior_mean_gap_days', 'prior_std_gap_days', 'spend_30d', 'purchase_count_90d', 'purchase_count_365d', 'is_first_purchase', 'purchase_year', 'purchase_dayofweek', 'customer_since_year']


**Conclusions:**
- With C=0.1, the L1 penalty keeps 29 of the 93 candidate columns on average and matches the reference (AP +0.0003, F1 −0.0017, both inside the noise).
- Unlike `SelectKBest`, it judges the columns together: of the purchase-volume windows it keeps only `purchase_count_90d`, `purchase_count_365d` and `spend_30d` in every split, and it adds columns that are weak on their own, such as `prior_mean_gap_days`, `is_first_purchase`, `purchase_year` and `customer_since_year`.
- Unlike RFE, its choice is stable: 17 columns are kept in all 7 splits, and only one of them is a dummy.
- C was fixed at 0.1 and not tuned, so a different penalty would keep a different number of columns.

---

## 3. Decision and Final Feature Set

The table puts every technique next to `reference_candidates`: the mean number of columns, the mean AP and F1, the number of columns kept in all 7 splits, and the paired difference with its verdict.

In [23]:
# Every technique next to the reference, on the same 7 splits
runs = {"reference_candidates": reference_candidates,
        "VarianceThreshold(0)": variance,
        **{f"SelectKBest(k={k})": run for k, run in kbest_runs.items()},
        **{f"RFE(k={k})": run for k, run in rfe.items()},
        "L1 (C=0.1)": l1}
rows = {}
for name, run in runs.items():
    kept = Counter(c for columns in run["kept"] for c in columns)
    row = {"columns": run["n"].mean(), "AP": run["AP"].mean(), "F1": run["F1"].mean(),
           "stable columns": sum(n == len(time_splits) for n in kept.values())}
    if name != "reference_candidates":
        for metric in ("AP", "F1"):
            r = paired_delta(run[metric], reference_candidates[metric])
            row[f"d{metric}"] = r["delta"]
            row[f"{metric} verdict"] = r["verdict"]
    rows[name] = row
summary = pd.DataFrame.from_dict(rows, orient="index").round({"columns": 1, "AP": 4, "F1": 4, "dAP": 4, "dF1": 4})
print(summary.to_string(na_rep=""))

                      columns      AP      F1  stable columns     dAP                AP verdict     dF1                F1 verdict
reference_candidates     93.0  0.8456  0.7419              81                                                                    
VarianceThreshold(0)     93.0  0.8456  0.7419              81  0.0000  identical on every split  0.0000  identical on every split
SelectKBest(k=10)        10.0  0.8317  0.7268               9 -0.0139                     worse -0.0152                     worse
SelectKBest(k=20)        20.0  0.8464  0.7473              17  0.0008          inside the noise  0.0054          inside the noise
SelectKBest(k=50)        50.0  0.8468  0.7433              31  0.0012          inside the noise  0.0014          inside the noise
RFE(k=10)                10.0  0.8349  0.7350               3 -0.0106          inside the noise -0.0070          inside the noise
RFE(k=20)                20.0  0.8397  0.7327               6 -0.0059                     

**Decision:** the feature selection strategy is **an L1-penalised logistic regression (C=0.1) on the 93 candidate columns**, refitted inside every training split.
- It matches the reference (AP +0.0003, F1 −0.0017, inside the noise) with 29 columns per split instead of 93.
- It judges the columns together, so it drops most of the repeated purchase-volume windows, and its choice is stable (17 columns kept in all 7 splits), unlike RFE.
- Its penalty was fixed in advance (C=0.1, the value used in the course practical), so, unlike the best k of `SelectKBest` or RFE, its result was not chosen on these splits.
- `SelectKBest` with k=20 also matches the reference, with fewer columns, but keeps all eight purchase-volume windows; RFE with k=20 and `SelectKBest` with k=10 are worse than the reference.

To obtain the final feature set, the strategy is fitted once on every training row: that is the set the final model uses. `fold_matrices(frame, frame, recipe)` prepares all the rows with the recipe learned on all of them.

In [24]:
# The chosen technique, fitted once on every training row: the final feature set
A_all, _, names_all = fold_matrices(frame, frame, recipe)
selector = SelectFromModel(
    LogisticRegression(l1_ratio=1, solver="liblinear", C=0.1, max_iter=1000,
                       random_state=RANDOM_STATE),
    threshold=1e-8).fit(A_all, frame[target])
final_features = [str(c) for c in names_all[selector.get_support()]]
print(f"{len(final_features)} of {len(names_all)} encoded columns kept:")
print(final_features)

39 of 106 encoded columns kept:
['payment_type_Maybe Tomorrow Payment 9', 'salesperson_VLOOKUP Master 4', 'commercial_zone_Decorative Dashboard Zone 4', 'document_series_Ctrl Z Series 3', 'commercial_region_Region RG17: We Will See Later', 'commercial_region_Region RG23: We Will See Later', 'commercial_region_Region RG27: We Will See Later', 'commercial_region_Region RG29: We Will See Later', 'island_(mainland)', 'island_Island IS1: The Pivot Archipelago', 'island_Island IS7: The Pivot Archipelago', 'Total a Pagar', 'Total Bruto', 'Total Imp_', 'Total Portes', 'Total Desc_ Linha', 'Custo', 'N_ Detalhes', 'invoice_count_current_purchase', 'days_since_previous_purchase', 'customer_tenure_days', 'prior_purchase_count', 'prior_mean_purchase_value', 'prior_std_purchase_value', 'prior_min_purchase_value', 'prior_mean_gap_days', 'prior_std_gap_days', 'spend_30d', 'purchase_count_90d', 'spend_180d', 'purchase_count_365d', 'is_first_purchase', 'purchase_year', 'purchase_month', 'purchase_day', 

**Conclusions:** On every training row, L1 keeps 39 of the 106 encoded columns. With more rows, the same C=0.1 penalises relatively less, so the final set is wider than the 29 columns of the average split. It still contains some near-duplicates, such as `Total a Pagar` and `Total Bruto`, whose weight the model can share between them.

In [25]:
# The strategy is saved as a rule, so the model notebooks refit it inside their own splits;
# the final list is kept for reference
strategy = {
    "candidate_columns": {"numeric": numeric, "categorical": categorical,
                          "dropped_identifier_like": identifier_like},
    "selector": {"method": "SelectFromModel with an L1-penalised LogisticRegression",
                 "l1_ratio": 1, "solver": "liblinear", "C": 0.1, "max_iter": 1000,
                 "threshold": 1e-8, "random_state": RANDOM_STATE},
    "fitted_per_split": True,
    "evaluation": "7 temporal splits; paired AP and F1 against the 93 candidate columns",
    "final_features": final_features,
}

output = Path("../../logs/feature_selection.json")
output.parent.mkdir(exist_ok=True)
output.write_text(json.dumps(strategy, indent=2), encoding="utf-8")
print(f"strategy saved to {output}")

strategy saved to ..\..\logs\feature_selection.json


The final feature set is also saved as a table for training the final model. It is built from the pre-processing team's file (`1_8_2.csv`, section 1.8.2), whose values for these 39 columns are identical to the preparation used in this notebook: it keeps `ID`, `customer_id`, the target and the columns L1 kept, with the team's column names (one dummy is named `payment_type_grouped_…`). Because both the preparation and the selection were fitted on every row, this file is for the final fit only: comparing models on the temporal splits must refit the strategy inside each split, as above.

In [26]:
# The team's prepared file (section 1.8.2), reduced to the columns L1 kept
prepared = pd.read_csv("../../data/preprocessing/1_8_2.csv")


def name_in_file(name):
    """1.8.1 encodes the grouped columns as '<column>_grouped_<level>'."""
    for column in recipe["rare"]:
        if name.startswith(column + "_"):
            return f"{column}_grouped_{name[len(column) + 1:]}"
    return name


final_table = prepared[["ID", "customer_id"] + [name_in_file(c) for c in final_features] + [target]]

outputFile = Path("../../data/featureselection/feature_selection_l1.csv")
outputFile.parent.mkdir(exist_ok=True)
final_table.to_csv(outputFile, index=False)
print(f"{final_table.shape[0]} rows x {final_table.shape[1]} columns saved to {outputFile}")

6469 rows x 42 columns saved to ..\..\data\featureselection\feature_selection_l1.csv


---

## 4. Conclusions

- **Starting point.** Removing the identifier-like columns (`city` and `postal_area`, which name customers, and `municipality` and `transport_method`, without which the model is better) and confirming that `random_noise` carries no information leaves 93 encoded candidate columns and improves F1 by +0.0157, better on 6 of the 7 splits. It is the only measurable gain in this notebook.
- **No technique improves on the candidates.** Every technique that does not lose stays inside the split-to-split noise: selection reduces the number of columns at no measurable cost rather than improving the prediction, as in the course practical.
- **How the families differ.** `SelectKBest` scores one column at a time and spends a small budget on repeated purchase-volume windows (worse with k=10); RFE is unstable across splits and favours rare dummies; L1 judges the columns together, drops most of the repetitions and keeps a stable set.
- **Strategy.** L1-penalised logistic regression (C=0.1) on the candidate columns, refitted inside every training split. Fitted on every training row, it keeps 39 of the 106 encoded columns; the strategy, that list and a table with those columns are saved for the model notebooks.
- **Limits.** The results come from 7 temporal splits with the same model; the best k of `SelectKBest` and RFE is optimistic because it was chosen on these splits; and L1 judges the columns through a linear model, so the strategy should be checked again when the non-linear models are fitted.